# neo4j 库操作展示

用来实验论文知识图（`docs/designs/graph_model.md`）上的操作方案与效果。

测试数据是 `_scratch/data_model_examples/` 下人工伪造的三批 yml，按真实顺序**一批一批**进库，
而不是一次性全部加载：先有 01 BERT；02 RAG 入库时要挂到 BERT 的既有节点上；03 GraphRAG 再挂到 RAG 上。

先写入一批伪造的测试数据，然后在这个图谱上检验一些检索、添加的效果（step9：按名称与 aliases 检索）。

流程会多次重跑：已入库且内容没变的批次 plan 出来全是"不变"，直接跳过写入。

## step1 预设参数

In [ ]:
import os
from pathlib import Path

DATA = Path.cwd() / "_scratch/data_model_examples"   # 三批测试数据，按入库顺序编号

# ── 连接参数：默认值对应 infra/neo4j/docker-compose.yml ──
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_AUTH = (os.environ.get("NEO4J_USER", "neo4j"), os.environ.get("NEO4J_PASSWORD", "password"))
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")

RESET = False   # True：清空整个库（DETACH DELETE 全部节点）后从空图重演；仅限这个库只放样例数据时使用

## step2 连接 neo4j，看库里已有什么

- 每个 label 建 `id` 唯一约束（`IF NOT EXISTS`，重跑无副作用），保证同一个 id 不会被建出两份；
- `library_status()` 打印当前库的内容，每批入库后都会再调一次。

In [ ]:
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import pandas as pd
import yaml
from neo4j import GraphDatabase, RoutingControl

# 空库时查 p.title 之类会收到“属性键不存在”的提示（UNRECOGNIZED 类），关掉这一类
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_disabled_classifications=["UNRECOGNIZED"])
driver.verify_connectivity()
print(NEO4J_URI, driver.get_server_info().agent)

def q(cypher: str, **params) -> list[dict]:
    """只读查询的简写。"""
    records, _, _ = driver.execute_query(cypher, params, database_=NEO4J_DB, routing_=RoutingControl.READ)
    return [r.data() for r in records]

# graph_model.md 里的 12 种 Node
LABELS = ["Paper", "Contribution", "Method", "MethodConcept", "Claim", "ClaimConcept",
          "Resource", "ResourceRecord", "Experiment", "Metric", "Condition", "ContentUnit"]
RESOURCE_KINDS = ["Dataset", "Benchmark", "Model", "CodeRepo", "Tool"]   # Resource 的次级 Label，恰好带一个
for label in LABELS:
    driver.execute_query(f"CREATE CONSTRAINT {label.lower()}_id IF NOT EXISTS "
                         f"FOR (n:{label}) REQUIRE n.id IS UNIQUE", database_=NEO4J_DB)

def library_status():
    labels = q("MATCH (n) UNWIND labels(n) AS l RETURN l, count(*) AS c ORDER BY l")
    nodes = q("MATCH (n) RETURN count(n) AS c")[0]["c"]   # 按节点计数；Resource 带两个 label，不能用 label 计数相加
    rels = q("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
    papers = q("MATCH (p:Paper) RETURN p.id AS id, p.title AS title, p.markdown_path IS NOT NULL AS full "
               "ORDER BY p.year, p.id")
    print(f"库内：节点 {nodes}，关系 {rels}")
    print("  " + ", ".join(f"{x['l']} {x['c']}" for x in labels) if labels else "  （空库）")
    for p in papers:
        print(f"  {'全文' if p['full'] else '仅标题'}  {p['id']:<16} {p['title'][:70]}")

if RESET:
    # 整库清空：样例 id 改过之后，按 yml 里的 id 删不掉旧节点。约束与索引保留。
    deleted = driver.execute_query("MATCH (n) DETACH DELETE n",
                                   database_=NEO4J_DB).summary.counters.nodes_deleted
    print(f"RESET：删除 {deleted} 个节点")

library_status()

## step3 入库的三个步骤

- **`load_batch`**：读 yml，只做和库无关的检查。节点 id 同时写进属性 `id`；label 和关系类型要拼进 Cypher 字符串，只放行 schema 内的 label 和标识符形式的关系类型。第一个 label 是 12 种主 Label 之一；Resource 还必须恰好带一个次级 Label（Dataset / Benchmark / Model / CodeRepo / Tool），其他节点不带次级 Label。
- **`plan_batch`**：拿本批涉及的全部 id 去库里查一遍，分类：
  - 节点：库里没有 → 新建；有但 yml 里某个属性值不同 → 更新（列出字段级差异）；否则不变。同一 id 在库里是别的 label → 报错。
  - 关系以 (from, type, to) 为身份，同样分新建 / 更新 / 不变。`DIFFERS_FROM` 等语义对称关系不分方向，库里有反向的一条也算已存在。
  - 本批只引用、不定义的 id 必须已在库里（前面批次入库的），否则是悬空引用，整批拒绝。
- **`apply_plan`**：只写新建和更新的部分，全部放进一个写事务；写完再 plan 一次，必须全部"不变"。

更新语义是 `SET n += props`：yml 里出现的属性覆盖，yml 里没写的属性保留。所以"先建仅有标题的 Paper、后补全"
这种增量补充直接成立；反过来，从 yml 里删掉一个属性**不会**删掉库里的值。

In [ ]:
SYMMETRIC = {"DIFFERS_FROM", "OVERLAPS_WITH", "CONTRADICTS"}   # graph_model.md 里的语义对称关系
REL_TYPE_RE = re.compile(r"^[A-Z][A-Z0-9_]*$")

def check_batch(raw: dict, name: str) -> dict:
    nodes, rels = {}, {}
    for n in raw.get("nodes") or []:
        assert n["id"] not in nodes, f"{name}: 批内重复定义 {n['id']}"
        main, *extra = n["labels"]
        assert main in LABELS, f"{name}: {n['id']} 的主 label {main!r} 不在 schema 里"
        if main == "Resource":
            assert len(extra) == 1 and extra[0] in RESOURCE_KINDS, \
                f"{name}: {n['id']} 需要恰好一个次级 Label {RESOURCE_KINDS}，实际是 {extra}"
        else:
            assert not extra, f"{name}: {n['id']} 不应带次级 Label {extra}"
        nodes[n["id"]] = {"labels": n["labels"], "props": {"id": n["id"], **(n.get("properties") or {})}}
    for r in raw.get("relationships") or []:
        key = (r["from"], r["type"], r["to"])
        assert REL_TYPE_RE.match(r["type"]), f"{name}: 关系类型不合法 {r['type']!r}"
        assert key not in rels, f"{name}: 批内重复关系 {key}"
        rels[key] = r.get("properties") or {}
    return {"name": name, "nodes": nodes, "rels": rels}

def load_batch(path: Path) -> dict:
    return check_batch(yaml.safe_load(path.read_text(encoding="utf-8")), path.name)

def diff_props(old: dict, new: dict) -> dict:
    """yml 里写了、且和库里不同的字段：{key: (库里的值, yml 的值)}。"""
    return {k: (old.get(k), v) for k, v in new.items() if old.get(k) != v}

In [ ]:
@dataclass
class Plan:
    batch: dict
    new_nodes: list = field(default_factory=list)
    upd_nodes: dict = field(default_factory=dict)    # id -> diff
    same_nodes: list = field(default_factory=list)
    referenced: dict = field(default_factory=dict)   # 本批只引用的既有节点 id -> labels
    new_rels: list = field(default_factory=list)
    upd_rels: dict = field(default_factory=dict)     # key -> diff
    same_rels: list = field(default_factory=list)

    @property
    def empty(self) -> bool:
        return not (self.new_nodes or self.upd_nodes or self.new_rels or self.upd_rels)

def plan_batch(b: dict) -> Plan:
    p = Plan(b)
    endpoints = {x for a, _, c in b["rels"] for x in (a, c)}
    # 测试规模小，这里按 id 全图查；规模大了应带上 label，走唯一约束的索引
    db = {r["id"]: r for r in q("MATCH (n) WHERE n.id IN $ids "
                                "RETURN n.id AS id, labels(n) AS labels, properties(n) AS props",
                                ids=sorted(set(b["nodes"]) | endpoints))}

    for nid, n in b["nodes"].items():
        if nid not in db:
            p.new_nodes.append(nid)
            continue
        assert set(db[nid]["labels"]) == set(n["labels"]), \
            f"{b['name']}: {nid} 在库里是 {db[nid]['labels']}，本批写成 {n['labels']}"
        d = diff_props(db[nid]["props"], n["props"])
        (p.upd_nodes.__setitem__(nid, d) if d else p.same_nodes.append(nid))

    outside = endpoints - set(b["nodes"])
    dangling = sorted(outside - set(db))
    assert not dangling, (f"{b['name']}: 关系引用了库里没有、本批也没定义的节点 {dangling}"
                          "——前面的批次是不是还没入库？")
    p.referenced = {i: db[i]["labels"] for i in sorted(outside)}

    keys = [{"from": a, "type": t, "to": c} for a, t, c in b["rels"]]
    existing = {(r["from"], r["type"], r["to"]): r["props"] for r in q(
        """UNWIND $keys AS k
           MATCH (a {id: k.from})-[r]-(c {id: k.to})
           WHERE type(r) = k.type AND (startNode(r) = a OR k.type IN $sym)
           RETURN k.from AS from, k.type AS type, k.to AS to, properties(r) AS props""",
        keys=keys, sym=sorted(SYMMETRIC))}
    for key, props in b["rels"].items():
        if key not in existing:
            p.new_rels.append(key)
        elif d := diff_props(existing[key], props):
            p.upd_rels[key] = d
        else:
            p.same_rels.append(key)
    return p

def short(v, n=60):
    s = str(v).replace("\n", " ")
    return s if len(s) <= n else s[:n] + "…"

def show_plan(p: Plan):
    b = p.batch
    print(f"== {b['name']} ==")
    print(f"节点  新建 {len(p.new_nodes)} · 更新 {len(p.upd_nodes)} · 不变 {len(p.same_nodes)}")
    print(f"关系  新建 {len(p.new_rels)} · 更新 {len(p.upd_rels)} · 不变 {len(p.same_rels)}")
    if p.new_nodes:
        c = Counter(b["nodes"][i]["labels"][0] for i in p.new_nodes)
        print("  新建节点：" + ", ".join(f"{k} {v}" for k, v in c.items()))
    if p.referenced:
        print(f"引用既有节点 {len(p.referenced)} 个（前面批次已入库，本批只往上挂关系）：")
        for i, labels in p.referenced.items():
            hooks = [f"{'←' if c == i else '→'} {t} {a if c == i else c}" for a, t, c in p.new_rels if i in (a, c)]
            print(f"  {i} {labels}  " + ("; ".join(hooks) if hooks else "（关系已存在）"))
    for i, d in p.upd_nodes.items():
        print(f"更新节点 {i}：")
        for k, (old, new) in d.items():
            print(f"    {k}: {short(old)} → {short(new)}")
    for key, d in p.upd_rels.items():
        print(f"更新关系 {key}：" + "; ".join(f"{k}: {short(o)} → {short(n)}" for k, (o, n) in d.items()))
    if p.empty:
        print("→ 与库内容一致，无需写入")

In [ ]:
def apply_plan(p: Plan):
    b = p.batch
    if p.empty:
        print(f"{b['name']}：无变化，跳过写入")
        return

    def work(tx):
        # 节点：按 label 组合分组，每组一条 UNWIND + MERGE
        groups = defaultdict(list)
        for nid in p.new_nodes + list(p.upd_nodes):
            n = b["nodes"][nid]
            groups[tuple(n["labels"])].append({"id": nid, "props": n["props"]})
        for labels, rows in groups.items():
            extra = "".join(f":{l}" for l in labels[1:])
            c = tx.run(f"UNWIND $rows AS row MERGE (n:{labels[0]} {{id: row.id}}) SET n += row.props"
                       + (f" SET n{extra}" if extra else "") + " RETURN count(n) AS c", rows=rows).single()["c"]
            assert c == len(rows)
        # 关系：按类型分组；语义对称的用无方向 MERGE，已有反向的一条就复用它
        groups = defaultdict(list)
        for key in p.new_rels + list(p.upd_rels):
            groups[key[1]].append({"from": key[0], "to": key[2], "props": b["rels"][key]})
        for t, rows in groups.items():
            arrow = "-" if t in SYMMETRIC else "->"
            c = tx.run(f"UNWIND $rows AS row MATCH (a {{id: row.from}}) MATCH (c {{id: row.to}}) "
                       f"MERGE (a)-[r:{t}]{arrow}(c) SET r += row.props RETURN count(r) AS c", rows=rows).single()["c"]
            assert c == len(rows), f"{t}: {len(rows)} 条里只写入 {c} 条（端点没匹配上）"

    with driver.session(database=NEO4J_DB) as s:
        s.execute_write(work)   # 单事务：中途出错整批回滚，不会留下半批
    print(f"{b['name']}：写入 节点 {len(p.new_nodes) + len(p.upd_nodes)}，关系 {len(p.new_rels) + len(p.upd_rels)}")

    again = plan_batch(b)   # 复核：写完之后库内容应与本批完全一致
    assert again.empty, "复核失败：写入后仍有差异"
    print("  ✓ 复核通过")

## step4 第 1 批：BERT

图谱的起点。本批没有引用任何外部节点；第一次运行时应全部是"新建"，之后重跑全部是"不变"。

In [ ]:
b01 = load_batch(DATA / "01_bert.yml")
p01 = plan_batch(b01)
show_plan(p01)

In [ ]:
apply_plan(p01)
library_status()

## step5 第 2 批：RAG

在 BERT 的基础上扩展。plan 里要看三件事：

- **挂接既有节点**：`pap_0001` BERT 论文（被 CITES）、`mth_0001` BERT 方法（被 HAS_METHOD reused）、`res_0002` BERT 预训练权重（`Resource:Model`，被 RELATES_TO、DESCRIBES）
  不在本批里定义，只被关系引用。它们的属性不动，但从此多了来自 RAG 的边；
- **仅有标题的 Paper**：`pap_0003`（DPR 论文）只带 title 入库，后面补全（见 step7）；
- **第一个跨论文命题** `ccp_0001`（外部可检索记忆提升知识密集型任务）。

如果 01 还没入库就跑这一步，plan 会因为悬空引用直接报错。

In [ ]:
b02 = load_batch(DATA / "02_rag.yml")
p02 = plan_batch(b02)
show_plan(p02)

In [ ]:
apply_plan(p02)
library_status()

## step6 第 3 批：GraphRAG

大量挂接 02 的既有节点：`pap_0002` RAG 论文（CITES）、`mth_0002` RAG 方法（DIFFERS_FROM，语义对称）、`mcp_0002` 检索增强生成
（SUBTYPE_OF、INSTANCE_OF）、`clm_0002` RAG 的开放域 QA 主张（被 QUALIFIES）、`ccp_0001`（被 REFINES）。

In [ ]:
b03 = load_batch(DATA / "03_graphrag.yml")
p03 = plan_batch(b03)
show_plan(p03)

In [ ]:
apply_plan(p03)
library_status()

## step7 更新既有节点：补全 `pap_0003`（DPR 论文）

前三批只有新增，没有改动已有节点的属性。这里在 notebook 里写一小批补丁（**不是**测试数据目录里的文件），演示后来拿到
DPR 的元数据、回头补全仅有标题的 Paper：同一个 id，plan 里是"更新"并列出字段差异，title 不写也不会被清掉。

In [ ]:
patch_dpr = check_batch({
    "nodes": [{
        "id": "pap_0003",   # DPR 论文，02 批只带 title 入库
        "labels": ["Paper"],
        "properties": {"year": 2020, "arxiv_id": "2004.04906", "paper_type": "method"},
    }],
}, name="patch_dpr（inline）")
p_patch = plan_batch(patch_dpr)
show_plan(p_patch)

In [ ]:
apply_plan(p_patch)
q("MATCH (p:Paper {id: 'pap_0003'}) RETURN properties(p) AS props")[0]["props"]

## step8 查看跨批次连起来的部分

三批各自入库，这里查的是只有放在一起才有的路径。

**命题细化**：03 的 Claim 所表达的命题 REFINES 02 的命题；03 的另一条 Claim 对 02 的 Claim 做 QUALIFIES。

In [ ]:
pd.DataFrame(q("""
MATCH (p2:Paper)-[:HAS_CLAIM]->(c2:Claim)-[:EXPRESSES]->(cc2:ClaimConcept)
      -[:REFINES]->(cc1:ClaimConcept)<-[:EXPRESSES]-(c1:Claim)<-[:HAS_CLAIM]-(p1:Paper)
RETURN left(p2.title, 30) AS paper, 'REFINES' AS rel, left(p1.title, 30) AS earlier_paper, cc2.text AS concept, cc1.text AS earlier_concept
UNION
MATCH (p2:Paper)-[:HAS_CLAIM]->(c2:Claim)-[:QUALIFIES]->(c1:Claim)<-[:HAS_CLAIM]-(p1:Paper)
RETURN left(p2.title, 30) AS paper, 'QUALIFIES' AS rel, left(p1.title, 30) AS earlier_paper, c2.text AS concept, c1.text AS earlier_concept
"""))

**多篇论文共用的方法**，以及方法所属类别（`INSTANCE_OF` / `SUBTYPE_OF`）。

In [ ]:
pd.DataFrame(q("""
MATCH (p:Paper)-[h:HAS_METHOD]->(m:Method)
WITH m, collect(left(p.title, 30) + ' (' + h.role + ')') AS papers
OPTIONAL MATCH (m)-[:INSTANCE_OF]->(c:MethodConcept)
OPTIONAL MATCH (c)-[:SUBTYPE_OF]->(sup:MethodConcept)
RETURN m.id AS id, m.name AS method, papers, c.name AS concept, sup.name AS super_concept
ORDER BY size(papers) DESC, id
"""))

## step9 检索演示：名称与 aliases

模拟抽取 Agent「查找已有实体」工具在样例图谱上的效果，查询写法对应 `docs/designs/graph_model.md` 的 Query 一节。
带 aliases 的 Node 是 Paper / Method / MethodConcept / Resource / Metric；aliases 不含主名称，也不是唯一键。
工具只返回候选和命中原因，是否复用由 Agent 判断。

- `find_exact`：名称 / 标题 / aliases 归一化（大小写、首尾空白）后精确相等；
- `find_fulltext`：全文索引 `entity_names` 召回写法相近的候选；
- `lookup`：先精确匹配，没有命中再走全文，附上候选的描述和直接相连的论文，供消歧。

In [ ]:
ALIAS_LABELS = ["Paper", "Method", "MethodConcept", "Resource", "Metric"]   # graph_model.md 中带 aliases 的 Node
LABEL_EXPR = "|".join(ALIAS_LABELS)
# Neo4j 不保证 labels(n) 的顺序；Resource 显示为 Resource:<次级 Label>
LABEL_OF = ("CASE WHEN {v}:Resource THEN 'Resource:' + [l IN labels({v}) WHERE l <> 'Resource'][0] "
            "ELSE labels({v})[0] END")

def find_exact(text: str) -> pd.DataFrame:
    """名称/标题/aliases 归一化后精确相等；hit 标出命中的是主名称还是哪个 alias。"""
    return pd.DataFrame(q(f"""
        WITH toLower(trim($text)) AS t
        MATCH (n:{LABEL_EXPR})
        WITH n, t, coalesce(n.name, n.title) AS name, coalesce(n.aliases, []) AS aliases
        WHERE toLower(trim(name)) = t OR ANY(a IN aliases WHERE toLower(trim(a)) = t)
        RETURN n.id AS id, {LABEL_OF.format(v='n')} AS label, name, aliases,
               CASE WHEN toLower(trim(name)) = t THEN 'name'
                    ELSE 'alias: ' + [a IN aliases WHERE toLower(trim(a)) = t][0] END AS hit
        ORDER BY label, id""", text=text), columns=["id", "label", "name", "aliases", "hit"])

for text in ["rag", "Graph RAG", "MLM", "em", "Natural Questions"]:
    df = find_exact(text)
    print(f"── {text!r}：精确命中 {len(df)} 个")
    if len(df):
        display(df)

- `"rag"`、`"Graph RAG"` 同时命中具体方法和方法类别：同一个称呼在不同论文里可能指方案，也可能指类别，alias 只负责召回，不能直接决定复用哪个。
- `"Natural Questions"` 没有精确命中：库里的 Resource 是 NQ 的开放域切分，样例有意没把原始数据集的名称记为它的 alias。这类只差一点的写法交给下面的全文检索召回。

In [ ]:
driver.execute_query(f"CREATE FULLTEXT INDEX entity_names IF NOT EXISTS "
                     f"FOR (n:{LABEL_EXPR}) ON EACH [n.name, n.title, n.aliases]", database_=NEO4J_DB)
driver.execute_query("CALL db.awaitIndexes(60)", database_=NEO4J_DB)

LUCENE_SPECIAL = re.compile(r'([+\-&|!(){}\[\]^"~*?:\\/])')

def find_fulltext(text: str, limit: int = 5) -> pd.DataFrame:
    """全文索引召回：name / title / aliases 分词后按相关度排序。text 里的 Lucene 特殊字符会被转义。"""
    return pd.DataFrame(q(f"""
        CALL db.index.fulltext.queryNodes('entity_names', $query, {{limit: $limit}}) YIELD node, score
        RETURN node.id AS id, {LABEL_OF.format(v='node')} AS label,
               coalesce(node.name, node.title) AS name, node.aliases AS aliases, round(score, 3) AS score""",
        query=LUCENE_SPECIAL.sub(r"\\\1", text), limit=limit),
        columns=["id", "label", "name", "aliases", "score"])

for text in ["Natural Questions", "graph rag summarization", "dense retriever", "Leiden community detection"]:
    print(f"── {text!r}")
    display(find_fulltext(text))

全文检索只看字面相似，召回结果里会混进只是字面相近的候选（例如 `"graph rag summarization"` 同时召回了 RAG 2020 和 GraphRAG 论文的向量检索基线），分数也不能直接当作同一对象的置信度。

`lookup` 把两层组合起来，再给每个候选补上描述和直接相连的论文及关系，这正是 Agent 做消歧时需要回看的上下文。`"BERT"` 同时命中方法（Method）和预训练权重（`Resource:Model`），二者靠 label 和描述区分；代码库 `google-research/bert` 名称不同，不在精确命中之列：

In [ ]:
def lookup(text: str, limit: int = 5) -> pd.DataFrame:
    """先精确匹配；没有命中再全文召回。附候选描述与直接相连的 Paper。"""
    exact = find_exact(text)
    how, ids = ("exact", list(exact.id)) if len(exact) else ("fulltext", list(find_fulltext(text, limit).id))
    ctx = {r["id"]: r for r in q(f"""
        MATCH (n) WHERE n.id IN $ids
        OPTIONAL MATCH (p:Paper)-[r]->(n)
        RETURN n.id AS id, {LABEL_OF.format(v='n')} AS label, coalesce(n.name, n.title) AS name,
               n.description AS description,
               collect(left(p.title, 24) + ' ' + type(r) + coalesce(' (' + r.role + ')', '')) AS papers""", ids=ids)}
    rows = [{"match": how, **ctx[i], "description": short(ctx[i]["description"], 50)} for i in ids]
    return pd.DataFrame(rows, columns=["match", "id", "label", "name", "description", "papers"])

for text in ["BERT", "RAG", "Natural Questions"]:
    print(f"── {text!r}")
    display(lookup(text))

这里还没覆盖的：

- **alias 追加**：入库仍是 `SET n += props`，yml 里的 aliases 会整体覆盖库里的值，在真实增量里需要改成追加去重；
- **候选上下文**：只附了直接相连的 Paper，MethodConcept 这类通过 `INSTANCE_OF` 间接关联论文的节点还看不到来源；
- **语义检索**：同义但字面不同的称呼（如中英文名称）目前只能靠 alias 覆盖，是否加向量检索看首轮遗漏情况再定。

In [ ]:
library_status()
driver.close()